# Multi-head attention

_Modern Deep Learning & AI_

**Run several attention heads at once. Each learns a different relationship, then you stitch them together.**

One attention pattern can only focus on one kind of relationship at a time.

     Multi-head attention runs several attentions side by side. Each head has its own query, key, and value vectors, so each learns to focus on something different.

---

This notebook builds multi-head attention one step at a time. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Build a multi-head attention module

PyTorch's `nn.MultiheadAttention` packs all the heads into one layer. We pick `d_model = 16` total dimensions and split them across `heads = 4`, so each head works in `16 / 4 = 4` dimensions. We seed the RNG first so the random weights are reproducible.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

batch = 1        # one sequence
seq = 4          # four tokens in that sequence
d_model = 16     # total embedding dimension
heads = 4        # 16 dims split into 4 heads of 4

mha = nn.MultiheadAttention(embed_dim=d_model, num_heads=heads, batch_first=True)

### Step 2 — Run self-attention on a toy input

We feed one tiny random sequence through the layer. In **self-attention** the query, key, and value all come from the same input `x`, so each token attends to every token in the sequence (including itself). The layer returns the new representations plus the attention weights.

In [ ]:
x = torch.randn(batch, seq, d_model)              # tiny synthetic input

# self-attention: query, key, value all come from x
out, weights = mha(x, x, x, need_weights=True)

print("output shape:", out.shape)                 # (1, 4, 16)
print("attn weights shape:", weights.shape)       # (1, 4, 4) averaged over heads

### Step 3 — Read off the shapes and properties

Each query row of the attention weights is a probability distribution over the keys, so it sums to ~1.0. The output keeps the full `d_model` dimension: every head produces a `d_model // heads`-sized slice, and those slices are concatenated back together.

In [ ]:
# each query row is a softmax over the keys, so it sums to ~1.0
print("each query row sums to:", weights[0, 0].sum().item())

# each head works in d_model // heads dims; concat returns to d_model
print("per-head dim:", d_model // heads)           # 4

## Visualize the data & results

_With 4 heads on the real sentence, how do different heads route the word it?_

### Step 1 — Build a sentence where it refers to animal

We use the classic ambiguous sentence and give each token a random embedding. Then we deliberately make the embedding for **"it"** a blend of mostly **"animal"** plus a little of its own noise, so a head that has learned to match similar vectors should route "it" toward "animal".

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# multi-head attention on a REAL sentence
tokens = ["The", "animal", "didnt", "cross", "the", "street",
          "because", "it", "was", "too", "tired"]

seq = len(tokens)
d_model = 16
heads = 4
d_head = d_model // heads

rng = np.random.default_rng(7)

# one random embedding per token
X = np.stack([np.random.default_rng(1000 + i).standard_normal(d_model)
              for i in range(seq)])

# make "it" (index 7) mostly "animal" (index 1) plus a little of itself
X[7] = 0.30 * X[7] + 0.95 * X[1]

### Step 2 — Compute one attention map per head

Each head gets its own random query and key projections, so each head measures a *different* notion of similarity. The `attn` helper scales the dot-product scores by `1/sqrt(d_head)`, subtracts the row max for numerical stability, then applies softmax so each query row becomes a distribution over the keys. We average the per-head maps to get the overall attention pattern.

In [ ]:
def attn(q, k):
    scores = q @ k.T / np.sqrt(q.shape[-1])        # scaled dot-product scores
    scores -= scores.max(axis=1, keepdims=True)    # subtract row max for stability
    weights = np.exp(scores)
    return weights / weights.sum(axis=1, keepdims=True)   # softmax over keys

maps = []
for h in range(heads):
    Wq = rng.standard_normal((d_model, d_head)) / np.sqrt(d_head)   # this head's query projection
    Wk = rng.standard_normal((d_model, d_head)) / np.sqrt(d_head)   # this head's key projection
    head_map = attn(X @ Wq, X @ Wk)
    maps.append(head_map)

maps = np.stack(maps)            # (heads, seq, seq)
avg = maps.mean(axis=0)          # attention averaged across the 4 heads

### Step 3 — Plot how the heads route the word it

The left panel shows the head-averaged attention map. The middle and right panels pick out the query row for **"it"** (index 7) under head 1 and head 2 — different heads put their weight on different tokens, which is the whole point of having multiple heads.

In [ ]:
it = 7        # query row for the word "it"

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

im = axes[0].imshow(avg, cmap="viridis", vmin=0, vmax=1)
axes[0].set_title("averaged over 4 heads")
fig.colorbar(im, ax=axes[0])

axes[1].bar(tokens, maps[0, it], color="#4ea1ff")
axes[1].set_title("it: head 1")

axes[2].bar(tokens, maps[1, it], color="#7ee787")
axes[2].set_title("it: head 2")

for a in axes[1:]:
    a.set_xticklabels(tokens, rotation=90)

plt.show()